# Model Evaluation and Comparison

This notebook is a centralized, read-only comparison of the four models. It **loads saved predictions and metrics only**; nothing is refit, and nothing here influences tuning.

## Plan (Prompt 5)

**1. Existing outputs and gaps.**
- Before this work, only the majority baseline and the teammate's RF wrote metrics, in different schemas. They had no probabilities (so no ROC/PR), no timing, and the baseline used a cwd-relative path.
- Now every model is run through `src/experiment.fit_and_evaluate`, which writes:
  - `results/metrics/{model}.json`: train/val/test metrics, threshold, timing, complexity, params, versions, hashes
  - `results/predictions/{model}_test_predictions.csv`: `y_true`, `y_prob`, `y_pred`, `label`, `attack_category`, `difficulty_level`, `is_novel`
- The teammate's original `random_forest.json` (a different preprocessor and a hand-picked configuration) is kept as a historical reference. The comparison uses `random_forest_optuna`, which was produced by the identical harness.

**2. Metrics and priorities.** The task is binary intrusion detection under prior shift: test is 57% attack vs 47% in train, and 29% of test attacks are novel.
- **Attack recall (detection rate)** comes first: a missed attack is the costly error.
- **FPR** is the alert-fatigue constraint.
- **Macro-F1** balances both classes and is insensitive to which class is the majority, so it is the tuning objective and headline score.
- **PR-AUC and ROC-AUC** are threshold-free ranking quality.
- MCC is an imbalance-robust summary.
- Accuracy is reported but not used for ranking: the majority baseline gets 43% accuracy with 0 recall.
- Inference time and complexity cover deployability.

**3. Changes to the model notebooks.** These are minimal: the baseline metrics path is fixed and the baseline runs through the harness; RF gets an appended Optuna section. LR and DT are new and use the harness directly.

**4–5. Structure and visuals.**
- metric table and grouped bars
- ROC and PR overlays with operating points
- side-by-side confusion matrices
- train/val/test generalization
- bootstrap confidence intervals
- per-category and novel-attack recall
- timing and complexity
- importance vs coefficients

**6. Safeguards.**
- Identical test rows, label mapping (`scalars/label_mapping.json`), train/val indices (`results/splits/`), CV folds and seed (42) for every model.
- Thresholds were chosen on validation.
- The test set was scored once per frozen model.

**7. Selection criteria.**
- Validation macro-F1 decides (development data only), subject to validation FPR ≤ 2%.
- If models are within 0.005, the simpler and faster model wins.
- Test results are then reported as evidence of generalization, not used to pick.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay, f1_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.config import load_config, set_all_seeds
CFG = load_config(); SEED = CFG["seed"]; rng = set_all_seeds(SEED)
sns.set_theme(style="whitegrid")
FIG = PROJECT_ROOT / "results" / "figures" / "evaluation"; FIG.mkdir(parents=True, exist_ok=True)
def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=110, bbox_inches="tight"); plt.show()

MODELS = {"majority_baseline": "Majority baseline", "logistic_regression": "Logistic Regression",
          "decision_tree": "Decision Tree", "random_forest_optuna": "Random Forest"}
COLORS = dict(zip(MODELS.values(), ["#8C8C8C", "#4C72B0", "#55A868", "#C44E52"]))
M = {v: json.loads((PROJECT_ROOT / "results" / "metrics" / f"{k}.json").read_text()) for k, v in MODELS.items()}
P = {v: pd.read_csv(PROJECT_ROOT / "results" / "predictions" / f"{k}_test_predictions.csv") for k, v in MODELS.items()}

# Fair-comparison guards: identical rows / labels / seed / data for every model
ref = next(iter(P.values()))
for name, df in P.items():
    assert len(df) == 22543 and (df["y_true"].to_numpy() == ref["y_true"].to_numpy()).all(), name
    assert (df["label"].to_numpy() == ref["label"].to_numpy()).all(), name
assert len({m["seed"] for m in M.values()}) == 1 and len({json.dumps(m["data_sha256"], sort_keys=True) for m in M.values()}) == 1
assert len({tuple(m["n_rows"].values()) for m in M.values()}) == 1
print("✓ identical test rows, labels, seed, data hashes and split sizes for all models")

## 1. Test metrics

In [ ]:
cols = ["threshold", "accuracy", "precision", "recall", "f1", "f1_macro", "f1_weighted", "mcc", "roc_auc", "pr_auc", "fpr", "fnr"]
test_tbl = pd.DataFrame({n: m["metrics"]["test"] for n, m in M.items()}).T[cols]
test_tbl.round(4).style.background_gradient(cmap="Greens", subset=["recall", "f1_macro", "roc_auc", "pr_auc", "mcc"]) \
    .background_gradient(cmap="Reds", subset=["fpr", "fnr"]).format("{:.4f}")

In [ ]:
show = ["accuracy", "precision", "recall", "f1_macro", "mcc", "roc_auc", "pr_auc"]
long = test_tbl[show].reset_index(names="model").melt(id_vars="model", var_name="metric")
fig, ax = plt.subplots(figsize=(15, 4.8))
sns.barplot(data=long, x="metric", y="value", hue="model", palette=COLORS, ax=ax)
for c in ax.containers: ax.bar_label(c, fmt="%.2f", fontsize=7)
ax.set_ylim(0, 1.08); ax.set_title("Test-set metrics at each model's validation-chosen threshold"); ax.legend(ncol=4, loc="upper center")
save(fig, "01_metric_bars")

## 2. Ranking quality and operating points

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for name, df in P.items():
    if df["y_prob"].nunique() < 2:  # constant baseline -> chance line
        continue
    fpr, tpr, _ = roc_curve(df.y_true, df.y_prob)
    axes[0].plot(fpr, tpr, c=COLORS[name], label=f"{name} (AUC {M[name]['metrics']['test']['roc_auc']:.3f})")
    t = M[name]["metrics"]["test"]; axes[0].scatter(t["fpr"], t["recall"], c=COLORS[name], s=70, edgecolor="k", zorder=3)
    pr, rc, _ = precision_recall_curve(df.y_true, df.y_prob)
    axes[1].plot(rc, pr, c=COLORS[name], label=f"{name} (AP {t['pr_auc']:.3f})")
    axes[1].scatter(t["recall"], t["precision"], c=COLORS[name], s=70, edgecolor="k", zorder=3)
axes[0].plot([0, 1], [0, 1], "k--", lw=.8, label="chance / majority baseline")
prev = ref.y_true.mean(); axes[1].axhline(prev, c="k", ls="--", lw=.8, label=f"baseline (prevalence {prev:.2f})")
axes[0].set(xlabel="False positive rate", ylabel="Attack recall", title="ROC (dots = chosen operating point)")
axes[1].set(xlabel="Attack recall", ylabel="Precision", title="Precision–Recall (dots = chosen operating point)")
axes[0].legend(loc="lower right"); axes[1].legend(loc="lower left")
fig.tight_layout(); save(fig, "02_roc_pr")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(21, 4.6))
for ax, (name, df) in zip(axes, P.items()):
    cm = confusion_matrix(df.y_true, df.y_pred, labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["normal", "attack"]).plot(ax=ax, cmap="Blues", values_format=",", colorbar=False)
    t = M[name]["metrics"]["test"]
    ax.set_title(f"{name}\nrecall {t['recall']:.3f} | FPR {t['fpr']:.3f}")
fig.tight_layout(); save(fig, "03_confusion_matrices")

## 3. Generalization: train vs validation vs test, with bootstrap CIs

In [ ]:
gen = pd.DataFrame({n: {s: m["metrics"][s]["f1_macro"] for s in ["train", "val", "test"]} for n, m in M.items()}).T
# 1,000 seeded bootstrap resamples of the test set (same indices for every model -> paired)
B = 1000
idx = rng.integers(0, len(ref), size=(B, len(ref)))
boot = {}
for name, df in P.items():
    yt, yp = df.y_true.to_numpy(), df.y_pred.to_numpy()
    boot[name] = np.array([f1_score(yt[i], yp[i], average="macro") for i in idx])
ci = pd.DataFrame({n: np.percentile(b, [2.5, 97.5]) for n, b in boot.items()}, index=["lo", "hi"]).T

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8), gridspec_kw={"width_ratios": [1.5, 1]})
gen.plot(kind="bar", ax=axes[0], rot=0, color=["#4C72B0", "#55A868", "#C44E52"])
for c in axes[0].containers: axes[0].bar_label(c, fmt="%.3f", fontsize=7)
axes[0].set_ylim(0, 1.08); axes[0].set_ylabel("macro-F1"); axes[0].set_title("In-distribution (train/val) vs official test")
trained = [n for n in MODELS.values() if n != "Majority baseline"]
for i, n in enumerate(trained):
    axes[1].errorbar(gen.loc[n, "test"], i, xerr=[[gen.loc[n, "test"] - ci.loc[n, "lo"]], [ci.loc[n, "hi"] - gen.loc[n, "test"]]],
                     fmt="o", c=COLORS[n], capsize=5, ms=8)
axes[1].set_yticks(range(len(trained))); axes[1].set_yticklabels(trained)
axes[1].set_title("Test macro-F1 with 95% bootstrap CI"); axes[1].set_xlabel("macro-F1")
fig.tight_layout(); save(fig, "04_generalization")

best_test = max(trained, key=lambda n: gen.loc[n, "test"])
diffs = pd.DataFrame({f"{best_test} − {n}": np.percentile(boot[best_test] - boot[n], [2.5, 50, 97.5])
                      for n in trained if n != best_test}, index=["2.5%", "median", "97.5%"]).T
display(gen.assign(val_minus_test=gen.val - gen.test).round(4)); diffs.round(4)

## 4. Where do models fail? Attack families and novel attacks

In [ ]:
rec = {}
for name, df in P.items():
    a = df[df.y_true == 1]
    r = a.groupby("attack_category")["y_pred"].mean()
    r["novel attacks"] = a.loc[a.is_novel == 1, "y_pred"].mean(); r["known attacks"] = a.loc[a.is_novel == 0, "y_pred"].mean()
    rec[name] = r
rec = pd.DataFrame(rec)
fig, ax = plt.subplots(figsize=(14, 4.5))
rec.drop(columns="Majority baseline").plot(kind="bar", ax=ax, rot=0, color=[COLORS[n] for n in trained])
for c in ax.containers: ax.bar_label(c, fmt="%.2f", fontsize=7)
n_att = ref[ref.y_true == 1]
counts = n_att.attack_category.value_counts().to_dict() | {"novel attacks": int(n_att.is_novel.sum()), "known attacks": int((1 - n_att.is_novel).sum())}
ax.set_xticklabels([f"{t.get_text()}\n(n={counts[t.get_text()]:,})" for t in ax.get_xticklabels()])
ax.set_ylim(0, 1.1); ax.set_ylabel("recall"); ax.set_title("Test recall by attack family and novelty")
fig.tight_layout(); save(fig, "05_recall_by_family")
rec.round(3)

## 5. Cost: training time, inference time and complexity

In [ ]:
def size_mb(name_key):
    p = PROJECT_ROOT / "models" / f"{name_key}_pipeline.joblib"
    return p.stat().st_size / 2**20 if p.exists() else np.nan
cost = pd.DataFrame({n: {"fit_s (full train)": m["timing"]["final_fit_seconds"],
                         "tuning_min": m.get("tuning", {}).get("tuning_seconds", 0) / 60,
                         "inference_us_per_row": m["timing"]["inference_us_per_row"],
                         "pipeline_MB (joblib, compressed)": size_mb(k)}
                     for (k, n), m in zip(MODELS.items(), M.values())}).T
cplx = pd.DataFrame({n: m["complexity"] for n, m in M.items()}).T
fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
for ax, col in zip(axes, ["fit_s (full train)", "inference_us_per_row", "pipeline_MB (joblib, compressed)"]):
    ax.bar(cost.index, cost[col], color=[COLORS[n] for n in cost.index]); ax.set_yscale("log"); ax.set_title(col + " (log)")
    ax.tick_params(axis="x", rotation=20)
    for i, v in enumerate(cost[col]): ax.text(i, v, f"{v:.3g}", ha="center", va="bottom", fontsize=8)
fig.tight_layout(); save(fig, "06_cost")
display(cost.round(3)); cplx

## 6. What drives each model? LR coefficients vs tree importances

In [ ]:
lr_pipe = joblib.load(PROJECT_ROOT / "models" / "logistic_regression_pipeline.joblib")
lr_names = [n.split("__", 1)[1] for n in lr_pipe[:-1].get_feature_names_out()]
imp = pd.DataFrame({
    "LR |coef| (scaled inputs)": pd.Series(np.abs(lr_pipe[-1].coef_[0]), index=lr_names),
    "DT impurity importance": pd.read_csv(PROJECT_ROOT / "results" / "tuning" / "decision_tree_feature_importance.csv", index_col=0).iloc[:, 0],
    "RF impurity importance": pd.read_csv(PROJECT_ROOT / "results" / "tuning" / "random_forest_optuna_feature_importance.csv", index_col=0).iloc[:, 0],
}).fillna(0)
imp = imp / imp.sum()                                        # normalise each to shares
top = imp.max(axis=1).sort_values(ascending=False).head(20).index
fig, ax = plt.subplots(figsize=(9, 9))
sns.heatmap(imp.loc[top], annot=True, fmt=".2f", cmap="YlOrRd", ax=ax, cbar=False)
ax.set_title("Normalised importance share — top-20 features (union)")
fig.tight_layout(); save(fig, "07_importance_vs_coefficients")
rank_corr = imp.rank(ascending=False).corr(method="spearman").round(2); rank_corr

## 7. Model selection

In [ ]:
val = pd.DataFrame({n: m["metrics"]["val"] for n, m in M.items()}).T
sel = pd.DataFrame({
    "val_f1_macro": val["f1_macro"], "val_fpr": val["fpr"], "test_f1_macro": test_tbl["f1_macro"],
    "test_recall": test_tbl["recall"], "test_fpr": test_tbl["fpr"],
    "inference_us_per_row": cost["inference_us_per_row"], "fit_s": cost["fit_s (full train)"],
}).drop("Majority baseline")
order_simplicity = {"Logistic Regression": 0, "Decision Tree": 1, "Random Forest": 2}
eligible = sel[sel.val_fpr <= 0.02]
top_val = eligible.val_f1_macro.max()
shortlist = eligible[eligible.val_f1_macro >= top_val - 0.005]
chosen = min(shortlist.index, key=lambda n: order_simplicity[n])
sel["eligible (val FPR ≤ 2%)"] = sel.index.isin(eligible.index)
sel["within 0.005 of best val"] = sel.index.isin(shortlist.index)
sel["selected"] = sel.index == chosen
print(f"Selected by the pre-declared rule: {chosen}")
print(f"Best test macro-F1 (reported, not used for selection): {sel.test_f1_macro.idxmax()}")
sel.round(4)

In [ ]:
verdict = pd.DataFrame([{**{"model": n}, **{k: M[n]["metrics"]["test"][k] for k in ["f1_macro", "recall", "fpr", "roc_auc", "pr_auc"]},
                         "novel_recall": M[n]["slices"]["recall_novel"], "params": json.dumps(M[n]["params"])} for n in M]).set_index("model")
verdict.to_csv(PROJECT_ROOT / "results" / "metrics" / "comparison_summary.csv")
sel.to_csv(PROJECT_ROOT / "results" / "metrics" / "model_selection.csv")
verdict.round(4)